In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [2]:
!pip install -q fastapi uvicorn python-multipart pyngrok nest-asyncio

In [4]:
!pip install -q langchain langchain-community langchain-core pypdf 

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 26.1 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 572.1/572.1 kB 16.2 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 35.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 1.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.29.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.3 which is incompatible.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
moviepy 1.0.3

In [3]:
!pip install -q transformers==4.52.4

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 71.0 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 21.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 51.9 MB/s eta 0:00:00:00:01


In [19]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "mistralai/Mistral-Nemo-Instruct-2407"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=torch.float16, device_map="auto")

def generate_text(prompt, max_length=1000, num_return_sequences=1):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    outputs = model.generate(
        **inputs,
        max_length=max_length,
        num_return_sequences=num_return_sequences,
        do_sample=True,
        top_k=50,
        top_p=0.95,
        temperature=0.7,
    )
    return [tokenizer.decode(output, skip_special_tokens=True) for output in outputs]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/622 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

...

model-00002-of-00005.safetensors:   0%|          | 0.00/4.91G [00:00<?, ?B/s]

model-00001-of-00005.safetensors:   0%|          | 0.00/4.87G [00:00<?, ?B/s]

model-00004-of-00005.safetensors:   0%|          | 0.00/4.91G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

In [5]:

from langchain_classic.output_parsers import StructuredOutputParser, ResponseSchema
from langchain_core.prompts import PromptTemplate
from langchain_community.document_loaders import PyPDFLoader
import torch
import re



/tmp/ipykernel_59/2605272742.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [6]:
pdf_path = "/kaggle/input/datasets/mahmoudmohamed2112/ahmed-cv/Resume(cv).pdf"
loader = PyPDFLoader(pdf_path)

documents = loader.load()


In [7]:
cv_text = "\n".join(
    doc.page_content
    for doc in documents
)
print(type(cv_text))

<class 'str'>


In [8]:
response_schemas = [
    ResponseSchema(
        name="full_name",
        description="The full name of the candidate as written in the CV."
    ),

    ResponseSchema(
        name="email",
        description="The candidate's email address."
    ),

    ResponseSchema(
        name="education",
        description=(
            "A list of education entries. "
            "Each entry must contain degree, institution, and year."
        )
    ),

    ResponseSchema(
        name="skills",
        description="A list of technical and professional skills mentioned in the CV."
    ),

    ResponseSchema(
        name="experience",
        description=(
            "A list of work experience entries. "
            "Each entry must contain role, company, and years."
        )
    )
]

In [9]:
output_parser = StructuredOutputParser.from_response_schemas(
    response_schemas
)
format_instructions = output_parser.get_format_instructions()

print(format_instructions)

The output should be a markdown code snippet formatted in the following schema, including the leading and trailing "```json" and "```":

```json
{
	"full_name": string  // The full name of the candidate as written in the CV.
	"email": string  // The candidate's email address.
	"education": string  // A list of education entries. Each entry must contain degree, institution, and year.
	"skills": string  // A list of technical and professional skills mentioned in the CV.
	"experience": string  // A list of work experience entries. Each entry must contain role, company, and years.
}
```


In [10]:
prompt_template = """
You are an HR Candidate Profile Parser.

Your task is to extract structured information from the candidate's CV.

Extract ONLY information that is actually present in the CV.

Do not invent information.

CV TEXT:
{cv_text}


{format_instructions}

Requirements:

1. full_name:
   Extract the candidate's full name.

2. email:
   Extract the candidate's email address.
   If it does not exist, return an empty string.

3. education:
   Return a list of objects containing:
   - degree
   - institution
   - year

4. skills:
   Return a list of skills.

5. experience:
   Return a list of objects containing:
   - role
   - company
   - years

Make sure the final response follows the required JSON format.
"""

In [20]:
prompt = PromptTemplate( 
    input_variables=["cv_text","format_instructions"] ,
    template=prompt_template
   
).format(cv_text=cv_text,format_instructions=format_instructions)

In [21]:
response = generate_text(prompt, max_length=2000)[0]
print(response)

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.



You are an HR Candidate Profile Parser.

Your task is to extract structured information from the candidate's CV.

Extract ONLY information that is actually present in the CV.

Do not invent information.

CV TEXT:
Ahmed Mohamed Samir
Mechatronics Engineer
Cairo, Egypt | 01280067615 | ahmedmohamedsamir553@gmail.com
PROFESSIONAL SUMMARY
Motivated Mechatronics Engineering graduate with comprehensive knowledge in mechanical design, embedded
systems, control systems, robotics, and industrial automation. Proven ability to integrate mechanics, electronics,
and programming to develop innovative and efficient engineering solutions. Experienced in real-world mechanical
engineering projects, maintenance, and quality control within construction, railway, and petroleum industries.
Skilled in PLC, SCADA, HMI, and industrial communication with strong problem-solving and leadership capabilities.
EDUCATION
Ain Shams University — Bachelor of Mechanical Engineering (Minor: Mechatronics)
Expected Graduati

In [22]:
def extract_json_block(text):
    pattern = r'```json\s*(.*?)\s*```'
    matches = re.findall(pattern, text, re.DOTALL)

    return f"```json\n{matches[-1]}\n```"

In [23]:
json_text = extract_json_block(response)

In [24]:
print(json_text)

```json
{
  "full_name": "Ahmed Mohamed Samir",
  "email": "ahmedmohamedsamir553@gmail.com",
  "education": [
    {
      "degree": "Bachelor of Mechanical Engineering (Minor: Mechatronics)",
      "institution": "Ain Shams University",
      "year": "Expected Graduation: June 2025"
    }
  ],
  "skills": [
    "Mechanical Design",
    "Embedded Systems",
    "Control Systems",
    "Robotics",
    "Industrial Automation",
    "PLC, SCADA, HMI",
    "Industrial Communication",
    "Leadership",
    "MEP (HVAC, Firefighting)",
    "Microsoft Office Suite",
    "Programming: Python, C++",
    "Problem Solving",
    "Teamwork"
  ],
  "experience": [
    {
      "role": "Intern",
      "company": "Cheiron Petroleum Corporation",
      "years": "2024"
    },
    {
      "role": "Intern",
      "company": "Egyptian National Railways",
      "years": "2023"
    },
    {
      "role": "Intern",
      "company": "Arab Contractors Company",
      "years": "2022"
    }
  ]
}
```


In [25]:
output_data = output_parser.parse(json_text)

In [26]:
print(output_data) 


{'full_name': 'Ahmed Mohamed Samir', 'email': 'ahmedmohamedsamir553@gmail.com', 'education': [{'degree': 'Bachelor of Mechanical Engineering (Minor: Mechatronics)', 'institution': 'Ain Shams University', 'year': 'Expected Graduation: June 2025'}], 'skills': ['Mechanical Design', 'Embedded Systems', 'Control Systems', 'Robotics', 'Industrial Automation', 'PLC, SCADA, HMI', 'Industrial Communication', 'Leadership', 'MEP (HVAC, Firefighting)', 'Microsoft Office Suite', 'Programming: Python, C++', 'Problem Solving', 'Teamwork'], 'experience': [{'role': 'Intern', 'company': 'Cheiron Petroleum Corporation', 'years': '2024'}, {'role': 'Intern', 'company': 'Egyptian National Railways', 'years': '2023'}, {'role': 'Intern', 'company': 'Arab Contractors Company', 'years': '2022'}]}


In [28]:
import json

import tempfile
import os

from langchain_community.document_loaders import PyPDFLoader


def extract_cv_from_pdf(pdf_path):

    # Load PDF
    loader = PyPDFLoader(pdf_path)
    documents = loader.load()

    # Convert PDF pages to one string
    cv_text = "\n".join(
        doc.page_content
        for doc in documents
    )

    # Create prompt
    prompt = PromptTemplate(
        input_variables=["cv_text", "format_instructions"],
        template=prompt_template
    ).format(
        cv_text=cv_text,
        format_instructions=format_instructions
    )

    # Generate response using your existing model
    response = generate_text(
        prompt,
        max_length=2000
    )[0]

    # Extract JSON code block
    pattern = r'```json\s*(.*?)\s*```'
    matches = re.findall(
        pattern,
        response,
        re.DOTALL
    )

    if not matches:
        raise ValueError(
            "Model did not return a valid JSON block"
        )

    json_text = f"```json\n{matches[-1]}\n```"

    # Parse using your existing LangChain parser
    output_data = output_parser.parse(json_text)

    return output_data

In [29]:
result = extract_cv_from_pdf(pdf_path)

print(result)

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


{'full_name': 'Ahmed Mohamed Samir', 'email': 'ahmedmohamedsamir553@gmail.com', 'education': [{'degree': 'Bachelor of Mechanical Engineering (Minor: Mechatronics)', 'institution': 'Ain Shams University', 'year': 'Expected Graduation: June 2025'}], 'skills': ['Mechanical Design', 'Embedded Systems', 'Control Systems', 'Robotics', 'Industrial Automation', 'PLC, SCADA, HMI', 'Industrial Communication', 'Leadership', 'MEP (HVAC, Firefighting)', 'Microsoft Office Suite', 'Programming: Python, C++', 'Problem Solving', 'Teamwork'], 'experience': [{'role': 'Intern', 'company': 'Cheiron Petroleum Corporation', 'years': '2024'}, {'role': 'Intern', 'company': 'Egyptian National Railways', 'years': '2023'}, {'role': 'Intern', 'company': 'Arab Contractors Company', 'years': '2022'}]}


In [30]:
from fastapi import FastAPI, UploadFile, File
from fastapi.responses import JSONResponse

app = FastAPI(
    title="CV Parser API",
    description="AI CV extraction API",
    version="1.0"
)


@app.get("/")
def home():

    return {
        "status": "online",
        "message": "CV Parser API is running"
    }


@app.get("/health")
def health():

    return {
        "status": "healthy"
    }


@app.post("/extract-cv")
async def extract_cv(
    file: UploadFile = File(...)
):

    try:

        # Check file type
        if file.content_type != "application/pdf":

            return JSONResponse(
                status_code=400,
                content={
                    "success": False,
                    "error": "Please upload a PDF file"
                }
            )

        # Read uploaded file
        file_bytes = await file.read()

        # Create temporary PDF
        with tempfile.NamedTemporaryFile(
            delete=False,
            suffix=".pdf"
        ) as temp_file:

            temp_file.write(file_bytes)
            temp_path = temp_file.name

        try:

            # Process CV
            result = extract_cv_from_pdf(
                temp_path
            )

        finally:

            # Delete temporary file
            if os.path.exists(temp_path):
                os.remove(temp_path)

        return {
            "success": True,
            "filename": file.filename,
            "data": result
        }

    except Exception as e:

        return JSONResponse(
            status_code=500,
            content={
                "success": False,
                "error": str(e)
            }
        )

In [31]:
import nest_asyncio
import uvicorn
import threading

nest_asyncio.apply()


def run_api():

    uvicorn.run(
        app,
        host="0.0.0.0",
        port=8000
    )


api_thread = threading.Thread(
    target=run_api,
    daemon=True
)

api_thread.start()

print("FastAPI started on port 8000")

FastAPI started on port 8000


INFO:     Started server process [59]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


In [32]:
from pyngrok import ngrok

NGROK_AUTH_TOKEN = "3K6gy9rUeppSPWs5MtgNQ6VrYlf_3zV6Dc8iGJ33aCs6xYVXM"

ngrok.set_auth_token(NGROK_AUTH_TOKEN)

public_url = ngrok.connect(
    8000,
    "http"
)

print("Public URL:")
print(public_url)

Public URL:                                                                                         
NgrokTunnel: "https://henchman-ground-elevator.ngrok-free.dev" -> "http://localhost:8000"


In [34]:
import requests

url = str("https://henchman-ground-elevator.ngrok-free.dev") + "/extract-cv"

with open(
    "/kaggle/input/datasets/mahmoudmohamed2112/ahmed-cv/Resume(cv).pdf",
    "rb"
) as f:

    response = requests.post(
        url,
        files={
            "file": (
                "Resume.pdf",
                f,
                "application/pdf"
            )
        }
    )

print(response.status_code)
print(response.json())

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


INFO:     34.73.234.105:0 - "POST /extract-cv HTTP/1.1" 200 OK
200
{'success': True, 'filename': 'Resume.pdf', 'data': {'full_name': 'Ahmed Mohamed Samir', 'email': 'ahmedmohamedsamir553@gmail.com', 'education': [{'degree': 'Bachelor of Mechanical Engineering (Minor: Mechatronics)', 'institution': 'Ain Shams University', 'year': 'Expected Graduation: June 2025'}], 'skills': ['Mechanical Design', 'Embedded Systems', 'Control Systems', 'Robotics', 'Industrial Automation', 'PLC, SCADA, HMI', 'Industrial Communication', 'Leadership', 'MEP (HVAC, Firefighting)', 'Microsoft Office Suite', 'Programming: Python, C++', 'Problem Solving', 'Teamwork'], 'experience': [{'role': 'Intern', 'company': 'Cheiron Petroleum Corporation', 'years': '2024'}, {'role': 'Intern', 'company': 'Egyptian National Railways', 'years': '2023'}, {'role': 'Intern', 'company': 'Arab Contractors Company', 'years': '2022'}]}}
INFO:     156.195.254.227:0 - "POST / HTTP/1.1" 405 Method Not Allowed
INFO:     156.195.254.227:0